In [20]:
import pg_gpu
import cupy as cp
import tskit
import msprime
import numpy as np
from pg_gpu.haplotype_matrix import HaplotypeMatrix

%load_ext autoreload
%autoreload 2

In [21]:
# do a simulation
ts = msprime.sim_ancestry(
    samples=10,
    sequence_length=5e5,
    recombination_rate=1e-8,
    population_size=10000,
    ploidy=2,
    discrete_genome=False
    )
ts = msprime.sim_mutations(ts, rate=1e-8, model="binary")
ts

In [23]:
ts.samples()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19], dtype=int32)

In [27]:
h = HaplotypeMatrix.from_ts(ts)
h

HaplotypeMatrix(shape=(20, 844), first_position=655.0, last_position=499691.0)

In [30]:
pi2 = h.pairwise_pi2()
pi2.shape

(844, 844)

In [156]:
%timeit ld_v = h.pairwise_LD_v()
%timeit -r 3 -n 1 ld = h.pairwise_LD()



246 μs ± 6.19 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
48.3 s ± 224 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [144]:
print(ts.diversity(mode="site"))
print(h.diversity(span_normalize=True))

0.0003820526315789475
0.0003820526315789474


In [142]:
print(ts.Tajimas_D())
print(h.Tajimas_D())

-0.21562971600837294
-0.21562971600837438


In [146]:
h.pairwise_r2()

array([[0.        , 0.09774436, 0.23076923, ..., 0.02834008, 0.09774436,
        0.05982906],
       [0.09774436, 0.        , 0.02255639, ..., 0.00277008, 0.00277008,
        0.00584795],
       [0.23076923, 0.02255639, 0.        , ..., 0.12280702, 0.02255639,
        0.02116402],
       ...,
       [0.02834008, 0.00277008, 0.12280702, ..., 0.        , 0.00277008,
        0.00584795],
       [0.09774436, 0.00277008, 0.02255639, ..., 0.00277008, 0.        ,
        0.00584795],
       [0.05982906, 0.00584795, 0.02116402, ..., 0.00584795, 0.00584795,
        0.        ]], shape=(143, 143))

# Comparison to moments

In [19]:
import moments.LD
import h5py
import pandas as pd

print(moments.LD.Util.moment_names(1))
b_bins = np.logspace(2, 6, 6)
print(b_bins)
vcf_path = "/home/adkern/pg_gpu/data/test.vcf"
ld_stats = moments.LD.Parsing.compute_ld_statistics(
    vcf_path,
    bp_bins=b_bins,
    use_genotypes=False,
)
print(ld_stats)

(['DD_0_0', 'Dz_0_0_0', 'pi2_0_0_0_0'], ['H_0_0'])
[1.00000000e+02 6.30957344e+02 3.98107171e+03 2.51188643e+04
 1.58489319e+05 1.00000000e+06]
 [2025-02-16 15:58:01] kept 172 total variants
 [2025-02-16 15:58:01] no recombination map provided, using physical distance
 [2025-02-16 15:58:01] No populations given, using all samples as one population.


 [2025-02-16 15:58:01] tallied two locus counts 0 of 172 positions
 [2025-02-16 15:58:01] computing DD_0_0
 [2025-02-16 15:58:01] computing Dz_0_0_0
 [2025-02-16 15:58:01] computing pi2_0_0_0_0
 [2025-02-16 15:58:01] computed sums
getting heterozygosity statistics
 [2025-02-16 15:58:01] No population file or population names given, assuming all samples as single pop.
{'bins': [(100.0, 630.957344480193), (630.957344480193, 3981.0717055349733), (3981.0717055349733, 25118.864315095823), (25118.864315095823, 158489.3192461114), (158489.3192461114, 1000000.0)], 'sums': [array([0.2378053 , 0.00082559, 0.52757138]), array([1.31633987, 0.58113175, 3.36546268]), array([ 6.2629343 ,  3.48218094, 18.61702786]), array([10.93599931,  8.35249398, 76.96406949]), array([ 20.21947024,  -3.02788098, 254.39315445]), array([53.45789474])], 'stats': (['DD_0_0', 'Dz_0_0_0', 'pi2_0_0_0_0'], ['H_0_0']), 'pops': ['ALL']}


In [32]:
print(ld_stats["stats"])
print(ld_stats["sums"])

(['DD_0_0', 'Dz_0_0_0', 'pi2_0_0_0_0'], ['H_0_0'])
[array([0.2378053 , 0.00082559, 0.52757138]), array([1.31633987, 0.58113175, 3.36546268]), array([ 6.2629343 ,  3.48218094, 18.61702786]), array([10.93599931,  8.35249398, 76.96406949]), array([ 20.21947024,  -3.02788098, 254.39315445]), array([53.45789474])]


In [38]:
h = HaplotypeMatrix.from_vcf(vcf_path)
h.tally_gpu_haplotypes()

array([[ 1,  0,  0, 19],
       [ 0,  1, 18,  1],
       [ 1,  0,  0, 19],
       ...,
       [ 6,  0,  6,  8],
       [ 6,  0,  3, 11],
       [ 9,  3,  0,  8]])

In [9]:
import allel
import numpy as np
from moments.LD import Parsing
from moments.LD.Parsing import _sparsify_haplotype_matrix
from pg_gpu.haplotype_matrix import HaplotypeMatrix
import cupy as cp
import sparse_tallying as spt

# Path to your VCF file
vcf_path = "/home/adkern/pg_gpu/data/test.vcf"

# Read the VCF using scikit-allel
vcf = allel.read_vcf(vcf_path)

# Get genotype data; note that the GT array has shape (num_variants, num_samples, ploidy)
genotypes = allel.GenotypeArray(vcf['calldata/GT'])
num_variants, num_samples, ploidy = genotypes.shape
assert ploidy == 2, "Expected diploid data"

# Convert the genotypes to a haplotype matrix.
# Each variant becomes a row, with two haplotypes per individual.
haplotypes = np.empty((num_variants, 2 * num_samples), dtype=genotypes.dtype)
haplotypes[:, :num_samples] = genotypes[:, :, 0]
haplotypes[:, num_samples:] = genotypes[:, :, 1]

# Create a sparse representation for the haplotypes using Moments’ internal utility.
# In Moments, _sparsify_haplotype_matrix expects a matrix of shape (variants, haplotypes)
# and returns a dictionary where each key (variant index) maps to a structure containing
# the set of sample indices with allele 1 (and possibly missing data info).
G_dict, any_missing = _sparsify_haplotype_matrix(haplotypes)
n_haplotypes = haplotypes.shape[1]

# For example, let’s compare the tally for the pair of variants (0, 1)
counts_moments = spt.tally_sparse_haplotypes(G_dict[0], G_dict[1], n_haplotypes, missing=any_missing)
print("Moments tally for variant pair (0, 1):", counts_moments)

# Now compare with our GPU version.
# Create a HaplotypeMatrix instance from the same VCF
h = HaplotypeMatrix.from_vcf(vcf_path)

# Our GPU method returns counts for every unique pair in an array of shape (#pairs, 4)
# using cupy.triu_indices (with ordering corresponding to (i, j) for i < j)
counts_gpu = h.tally_gpu_haplotypes()  # returns a CuPy array

# We need to find the row corresponding to the pair (0, 1)
# Get the upper-triangular indices from the haplotype matrix
idx_i, idx_j = cp.triu_indices(h.num_variants, k=1)
pairs = np.column_stack((idx_i, idx_j))
target_index = np.where((pairs[:, 0] == 0) & (pairs[:, 1] == 1))[0][0]

# Move the counts to CPU for comparison
counts_gpu_pair = counts_gpu[target_index]
print("GPU tally for variant pair (0, 1):", counts_gpu_pair)

Moments tally for variant pair (0, 1): (1, 0, 0, 19)
GPU tally for variant pair (0, 1): [ 1  0  0 19]


In [10]:
import allel
import numpy as np
from moments.LD.Parsing import _sparsify_haplotype_matrix
import sparse_tallying as spt  # Moments' LD extension for tallying haplotypes
from pg_gpu.haplotype_matrix import HaplotypeMatrix
import cupy as cp

# Path to your VCF file
vcf_path = "/home/adkern/pg_gpu/data/test.vcf"

# Read the VCF using scikit-allel
vcf = allel.read_vcf(vcf_path)

# Get genotype data; note that the GT array has shape (num_variants, num_samples, ploidy)
genotypes = allel.GenotypeArray(vcf['calldata/GT'])
num_variants, num_samples, ploidy = genotypes.shape
assert ploidy == 2, "Expected diploid data"

# Construct a haplotype matrix where each row corresponds to a variant and 
# each column to one haplotype (as expected by _sparsify_haplotype_matrix).
haplo_for_sparse = np.empty((num_variants, 2 * num_samples), dtype=genotypes.dtype)
haplo_for_sparse[:, :num_samples] = genotypes[:, :, 0]
haplo_for_sparse[:, num_samples:] = genotypes[:, :, 1]

n_haplotypes = haplo_for_sparse.shape[1]
print("Sparse method n_haplotypes:", n_haplotypes)

# Create a sparse representation using Moments’ internal utility.
# G_dict is a dictionary mapping each variant index to its sparse representation.
G_dict, any_missing = _sparsify_haplotype_matrix(haplo_for_sparse)

# --- Compute tallies using the Moments implementation for all variant pairs ---
pairs_moments = []
counts_moments_all = []
for i in range(num_variants - 1):
    for j in range(i + 1, num_variants):
        counts = spt.tally_sparse_haplotypes(G_dict[i], G_dict[j], n_haplotypes, missing=any_missing)
        pairs_moments.append((i, j))
        counts_moments_all.append(counts)
counts_moments_all = np.array(counts_moments_all)  # shape: (num_pairs, 4)

# --- Compute tallies using the GPU method from HaplotypeMatrix ---
h = HaplotypeMatrix.from_vcf(vcf_path)
counts_gpu = h.tally_gpu_haplotypes()  # returns a CuPy array
counts_gpu = counts_gpu.get()          # convert to NumPy array

# Determine the GPU pair ordering using cupy.triu_indices on the variant axis.
idx_i, idx_j = cp.triu_indices(h.num_variants, k=1)
idx_i = idx_i.get()
idx_j = idx_j.get()
pairs_gpu = np.column_stack((idx_i, idx_j))

# --- Compare the results pair-by-pair ---
all_match = True
num_mismatch = 0
for pair, moment_tally in zip(pairs_moments, counts_moments_all):
    # Locate the corresponding GPU pair
    cond = (pairs_gpu[:, 0] == pair[0]) & (pairs_gpu[:, 1] == pair[1])
    idx_arr = np.where(cond)[0]
    if idx_arr.size == 0:
        print(f"Pair {pair} not found in GPU ordering.")
        continue
    gpu_tally = counts_gpu[idx_arr[0]]
    if not np.array_equal(moment_tally, gpu_tally):
        print(f"Mismatch for pair {pair}: Moments tally = {moment_tally}, GPU tally = {gpu_tally}")
        all_match = False
        num_mismatch += 1

if all_match:
    print("All variant pairs match between Moments and GPU tallies!")
else:
    print(f"{num_mismatch} variant pairs mismatched between Moments and GPU tallies.")

# Optionally, print a sample of the first several comparisons for inspection.
print("\nSample tallies:")
sample_count = min(5, counts_moments_all.shape[0])
for i in range(sample_count):
    print(f"Pair {pairs_moments[i]}: Moments = {counts_moments_all[i]}, GPU = {counts_gpu[i]}")

Sparse method n_haplotypes: 20
All variant pairs match between Moments and GPU tallies!

Sample tallies:
Pair (0, 1): Moments = [ 1  0  0 19], GPU = [ 1  0  0 19]
Pair (0, 2): Moments = [ 0  1 18  1], GPU = [ 0  1 18  1]
Pair (0, 3): Moments = [ 1  0  0 19], GPU = [ 1  0  0 19]
Pair (0, 4): Moments = [ 0  1  2 17], GPU = [ 0  1  2 17]
Pair (0, 5): Moments = [ 0  1  3 16], GPU = [ 0  1  3 16]


In [18]:
import allel
import numpy as np
from moments.LD.Parsing import _sparsify_haplotype_matrix
import sparse_tallying as spt  # Moments’ LD extension for tallying haplotypes
from pg_gpu.haplotype_matrix import HaplotypeMatrix
import cupy as cp

# ----------------------------
# PREPARATION: Build data structures once
# ----------------------------
vcf_path = "/home/adkern/pg_gpu/data/test2.vcf"

# Read VCF using scikit-allel.
vcf = allel.read_vcf(vcf_path)
genotypes = allel.GenotypeArray(vcf['calldata/GT'])
num_variants, num_samples, ploidy = genotypes.shape
assert ploidy == 2, "Expected diploid data"

# Construct haplotype matrix (variants x haplotypes)
haplo_for_sparse = np.empty((num_variants, 2 * num_samples), dtype=genotypes.dtype)
haplo_for_sparse[:, :num_samples] = genotypes[:, :, 0]
haplo_for_sparse[:, num_samples:] = genotypes[:, :, 1]
n_haplotypes = haplo_for_sparse.shape[1]
print("Sparse method n_haplotypes:", n_haplotypes)
print("number of variants:", num_variants)
# Compute Moments’ sparse representation.
G_dict, any_missing = _sparsify_haplotype_matrix(haplo_for_sparse)

# Build the GPU version once.
h = HaplotypeMatrix.from_vcf(vcf_path)

# ----------------------------
# DEFINE FUNCTIONS FOR TIMING
# ----------------------------
def run_moments_tally():
    """
    Loop through all variant pairs and tally using the Moments compiled function.
    """
    for i in range(num_variants - 1):
        for j in range(i + 1, num_variants):
            _ = spt.tally_sparse_haplotypes(G_dict[i], G_dict[j], n_haplotypes, missing=any_missing)

def run_gpu_tally():
    """
    Compute the GPU tallies and then force synchronization.
    """
    counts_gpu = h.tally_gpu_haplotypes()   # returns a CuPy array
    cp.cuda.Stream.null.synchronize()       # Force synchronization for accurate timing
    return counts_gpu

# ----------------------------
# PERFORMANCE MEASUREMENT: Using %timeit
# ----------------------------

# Run timeit with the -o option to capture the results.
moments_result = %timeit -o run_moments_tally()
gpu_result = %timeit -o run_gpu_tally()

# Extracting the best execution times (in seconds) from the results.
moments_time = moments_result.best
gpu_time = gpu_result.best

print("Moments best execution time: {:.6f} seconds".format(moments_time))
print("GPU best execution time: {:.6f} seconds".format(gpu_time))

# Calculate the speedup: how many times faster the GPU version is.
speedup = moments_time / gpu_time
print("GPU is {:.2f} times faster than Moments".format(speedup))

Sparse method n_haplotypes: 20
number of variants: 14205
1min 6s ± 139 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
42.9 ms ± 48.9 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
Moments best execution time: 66.502864 seconds
GPU best execution time: 0.042842 seconds
GPU is 1552.28 times faster than Moments


In [38]:
import numpy as np
import moments.LD.Parsing as mParsing
from pg_gpu.haplotype_matrix import HaplotypeMatrix

# Define bp bin boundaries (for example, logarithmically spaced)
bp_bins = np.logspace(2, 6, 6)
print("Base pair bins:", bp_bins)

# Use an existing VCF file (adjust path as needed)
vcf_path = "/home/adkern/pg_gpu/data/test.vcf"

# -----------------------------
# Compute LD statistics using Moments.
# Note: Moments returns a dictionary with keys "bins" and "sums" among others.
# The "sums" are raw sums that need to be averaged later.
# -----------------------------
print("Computing LD statistics with Moments...")
ld_stats_moments = mParsing.compute_ld_statistics(vcf_path, bp_bins=bp_bins, use_genotypes=False)
print("Moments bin definitions:", ld_stats_moments["bins"])
print("Moments raw sums:")
for s in ld_stats_moments["sums"]:
    print(" ", s)
# -----------------------------
# Compute LD statistics using our GPU-based implementation, returning raw sums.
# -----------------------------
print("\nComputing raw LD statistics with GPU-based HaplotypeMatrix...")
h_gpu = HaplotypeMatrix.from_vcf(vcf_path)
ld_stats_gpu_raw = h_gpu.compute_ld_statistics_gpu(bp_bins=bp_bins, missing=False, raw=True)
print("GPU LD raw sums per bin:")
for bin_key, stats in ld_stats_gpu_raw.items():
    # match the number of decimal places to the Moments output
    stats = tuple(round(s, 7) for s in stats)
    print(f"  Bin {bin_key}: {stats}")

Base pair bins: [1.00000000e+02 6.30957344e+02 3.98107171e+03 2.51188643e+04
 1.58489319e+05 1.00000000e+06]
Computing LD statistics with Moments...
 [2025-02-16 21:58:37] kept 172 total variants
 [2025-02-16 21:58:37] no recombination map provided, using physical distance
 [2025-02-16 21:58:37] No populations given, using all samples as one population.


 [2025-02-16 21:58:37] tallied two locus counts 0 of 172 positions
 [2025-02-16 21:58:37] computing DD_0_0
 [2025-02-16 21:58:37] computing Dz_0_0_0
 [2025-02-16 21:58:37] computing pi2_0_0_0_0
 [2025-02-16 21:58:37] computed sums
getting heterozygosity statistics
 [2025-02-16 21:58:37] No population file or population names given, assuming all samples as single pop.
Moments bin definitions: [(100.0, 630.957344480193), (630.957344480193, 3981.0717055349733), (3981.0717055349733, 25118.864315095823), (25118.864315095823, 158489.3192461114), (158489.3192461114, 1000000.0)]
Moments raw sums:
  [0.2378053  0.00082559 0.52757138]
  [1.31633987 0.58113175 3.36546268]
  [ 6.2629343   3.48218094 18.61702786]
  [10.93599931  8.35249398 76.96406949]
  [ 20.21947024  -3.02788098 254.39315445]
  [53.45789474]

Computing raw LD statistics with GPU-based HaplotypeMatrix...
GPU LD raw sums per bin:
  Bin (100.0, 630.957344480193): (0.2378053, 0.0008256, 0.5275714)
  Bin (630.957344480193, 3981.071705

In [42]:
import numpy as np
import moments.LD.Parsing as mParsing
from pg_gpu.haplotype_matrix import HaplotypeMatrix

# Define base-pair bin boundaries and the VCF file to use.
bp_bins = np.logspace(2, 6, 6)
vcf_path = "/home/adkern/pg_gpu/data/test2.vcf"

# ----------------------------
# Profile the Moments implementation.
# ----------------------------
print("Timing Moments compute_ld_statistics...")
moments_time_result = %timeit -o mParsing.compute_ld_statistics(vcf_path, bp_bins=bp_bins, use_genotypes=False, report=False)
moments_best = moments_time_result.best
print("Moments best execution time: {:.6f} seconds".format(moments_best))

# ----------------------------
# Profile the GPU-based implementation.
# ----------------------------
print("\nTiming GPU compute_ld_statistics_gpu (raw output)...")
# Create an instance from the VCF.
h_gpu = HaplotypeMatrix.from_vcf(vcf_path)
gpu_time_result = %timeit -o h_gpu.compute_ld_statistics_gpu(bp_bins=bp_bins, missing=False, raw=True)
gpu_best = gpu_time_result.best
print("GPU best execution time: {:.6f} seconds".format(gpu_best))

# ----------------------------
# Compute and print the speedup.
# ----------------------------
speedup = moments_best / gpu_best
print("\nGPU is {:.2f} times faster than Moments compute_ld_statistics.".format(speedup))

Timing Moments compute_ld_statistics...
1.37 s ± 2.96 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Moments best execution time: 1.369509 seconds

Timing GPU compute_ld_statistics_gpu (raw output)...
12.3 ms ± 6.54 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
GPU best execution time: 0.012325 seconds

GPU is 111.12 times faster than Moments compute_ld_statistics.
